In [1]:
import sys

print(sys.version)
print(sys.executable)

3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
D:\practica_bfs\bfs_env\Scripts\python.exe


In [2]:
import random
from collections import deque

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from IPython.display import HTML, display


# ==========================================================
# 1. CONFIGURACIÓN
# ==========================================================

SEED = 42
DIM = 31
MAX_TICKS = 300
INTERVAL_MS = 150

random.seed(SEED)
np.random.seed(SEED)

plt.rcParams["animation.embed_limit"] = 100  # MB

start_pos = (1, 1)
target_pos = (DIM - 2, DIM - 2)

directions = [
    (-1, 0),
    (1, 0),
    (0, -1),
    (0, 1)
]


# ==========================================================
# 2. SUBRUTINA ORIGINAL DEL PROFESOR
# ==========================================================

def calculate_bfs_path(matrix, start, target, enemies):
    rows, cols = matrix.shape
    forbidden_zones = set()
    for ex, ey in enemies:
        forbidden_zones.add((ex, ey))
        for dx, dy in directions:
            nx, ny = ex + dx, ey + dy
            if 0 <= nx < rows and 0 <= ny < cols:
                forbidden_zones.add((nx, ny))

    queue = deque([(start, [start])])
    visited = {start}

    while queue:
        current, path = queue.popleft()
        if current == target:
            return path

        for dr, dc in directions:
            nr, nc = current[0] + dr, current[1] + dc
            if (
                0 <= nr < rows
                and 0 <= nc < cols
                and matrix[nr, nc] == 0
            ):
                if (
                    (nr, nc) not in forbidden_zones
                    or (nr, nc) == target
                ):
                    if (nr, nc) not in visited:
                        visited.add((nr, nc))
                        queue.append(
                            ((nr, nc), path + [(nr, nc)])
                        )
    return [start]


# ==========================================================
# 3. GENERACIÓN DEL LABERINTO
# ==========================================================

def generate_maze():
    matrix = np.zeros((DIM, DIM), dtype=int)

    # Perímetro
    matrix[0, :] = 1
    matrix[-1, :] = 1
    matrix[:, 0] = 1
    matrix[:, -1] = 1

    # Obstáculos estructurales
    for row in range(1, DIM - 1):
        for col in range(1, DIM - 1):
            if row % 6 == 0 and col % 6 == 0:
                matrix[row, col] = 1

    # Muros aleatorios
    for _ in range(180):
        row = random.randint(1, DIM - 2)
        col = random.randint(1, DIM - 2)

        if matrix[row, col] == 0:
            length = random.randint(2, 6)
            orientation = random.choice(["h", "v"])

            if orientation == "h" and col + length < DIM - 1:
                if np.sum(matrix[row, col:col + length]) == 0:
                    matrix[row, col:col + length] = 1

            elif orientation == "v" and row + length < DIM - 1:
                if np.sum(matrix[row:row + length, col]) == 0:
                    matrix[row:row + length, col] = 1

    # Ajuste de densidad
    for _ in range(80):
        row = random.randint(1, DIM - 2)
        col = random.randint(1, DIM - 2)
        matrix[row, col] = 0

    matrix[1:3, 1:3] = 0
    matrix[DIM - 4:DIM - 1, DIM - 4:DIM - 1] = 0

    return matrix


# Reintentar para evitar comenzar con una meta inaccesible.
maze_base = None
generation_attempts = 0
corridor_added = False

for attempt in range(1, 51):
    candidate = generate_maze()
    initial_path = calculate_bfs_path(
        candidate, start_pos, target_pos, []
    )

    if len(initial_path) > 1:
        maze_base = candidate
        generation_attempts = attempt
        break

# Si ningún intento es conectado, abrir un corredor inicial.
if maze_base is None:
    maze_base = candidate
    maze_base[1, 1:DIM - 1] = 0
    maze_base[1:DIM - 1, DIM - 2] = 0
    generation_attempts = 50
    corridor_added = True


# ==========================================================
# 4. POSICIONES VÁLIDAS PARA LOS ENEMIGOS
# ==========================================================

def reachable_cells(matrix, start):
    queue = deque([start])
    visited = {start}

    while queue:
        row, col = queue.popleft()

        for dr, dc in directions:
            nr, nc = row + dr, col + dc

            if (
                0 <= nr < DIM
                and 0 <= nc < DIM
                and matrix[nr, nc] == 0
                and (nr, nc) not in visited
            ):
                visited.add((nr, nc))
                queue.append((nr, nc))

    return visited


reachable = reachable_cells(maze_base, start_pos)

preferred_positions = [
    (1, DIM - 2),
    (29, 1),
    (15, 15),
    (15, 5),
    (5, 5)
]

enemy_positions = []

for preferred in preferred_positions:
    candidates = [
        cell for cell in reachable
        if cell not in enemy_positions
        and cell != target_pos
        and abs(cell[0] - start_pos[0])
        + abs(cell[1] - start_pos[1]) >= 6
    ]

    if not candidates:
        raise RuntimeError(
            "No hay suficientes celdas válidas para los enemigos."
        )

    # Si la posición original es válida, se conserva.
    # Si no, se usa la celda accesible más cercana.
    chosen = min(
        candidates,
        key=lambda cell: (
            abs(cell[0] - preferred[0])
            + abs(cell[1] - preferred[1]),
            cell[0],
            cell[1]
        )
    )

    enemy_positions.append(chosen)


# ==========================================================
# 5. SIMULACIÓN Y REGISTRO DE ESTADOS
# ==========================================================

def hazard_cells(enemies):
    hazards = set()

    for row, col in enemies:
        hazards.add((row, col))

        for dr, dc in directions:
            nr, nc = row + dr, col + dc

            if (
                0 <= nr < DIM
                and 0 <= nc < DIM
                and maze_base[nr, nc] == 0
            ):
                hazards.add((nr, nc))

    return hazards


def simulate():
    agent = start_pos
    enemies = enemy_positions.copy()

    records = []
    blocked_ticks = 0
    route_changes = 0
    outcome = "Límite de pasos"

    def record(tick, path, status):
        records.append({
            "tick": tick,
            "agent": agent,
            "enemies": enemies.copy(),
            "path": path.copy(),
            "status": status
        })

    current_path = calculate_bfs_path(
        maze_base, agent, target_pos, enemies
    )
    record(0, current_path, "Estado inicial")

    for tick in range(1, MAX_TICKS + 1):
        if agent == target_pos:
            outcome = "Meta alcanzada"
            break

        # Ruta anterior, desde la posición actual.
        old_path = calculate_bfs_path(
            maze_base, agent, target_pos, enemies
        )

        # Enemigos: movimiento aleatorio de una celda.
        for index, (row, col) in enumerate(enemies):
            possible_moves = []

            for dr, dc in directions:
                nr, nc = row + dr, col + dc

                if (
                    0 <= nr < DIM
                    and 0 <= nc < DIM
                    and maze_base[nr, nc] == 0
                    and (nr, nc) != target_pos
                ):
                    possible_moves.append((nr, nc))

            if possible_moves:
                enemies[index] = random.choice(possible_moves)

        # Comprobar colisión antes de permitir la respuesta.
        if agent in enemies:
            outcome = "Colisión con enemigo"
            record(tick, [], outcome)
            break

        new_path = calculate_bfs_path(
            maze_base, agent, target_pos, enemies
        )

        # Cambio de ruta por cambio de la configuración enemiga.
        if (
            len(old_path) > 1
            and len(new_path) > 1
            and old_path != new_path
        ):
            route_changes += 1

        if len(new_path) > 1:
            agent = new_path[1]
            remaining_path = new_path[1:]
            status = "Avanzando"
        else:
            blocked_ticks += 1
            remaining_path = [agent]
            status = "Sin ruta disponible"

        if agent == target_pos:
            outcome = "Meta alcanzada"
            status = outcome

        record(tick, remaining_path, status)

        if agent == target_pos:
            break

    return records, blocked_ticks, route_changes, outcome


records, blocked_ticks, route_changes, outcome = simulate()


# ==========================================================
# 6. ANIMACIÓN EN JUPYTER
# ==========================================================

fig, ax = plt.subplots(figsize=(8, 8))

cmap = ListedColormap([
    "#F4F6F7",  # Libre
    "#34495E"   # Muro
])

ax.imshow(maze_base, cmap=cmap, vmin=0, vmax=1)
ax.axis("off")

route_line, = ax.plot(
    [], [],
    color="#3498DB",
    linewidth=2,
    alpha=0.8,
    zorder=2
)

hazard_points = ax.scatter(
    [], [],
    color="#E74C3C",
    marker="s",
    s=100,
    alpha=0.18,
    zorder=3
)

enemy_points = ax.scatter(
    [], [],
    color="#E74C3C",
    s=65,
    zorder=4
)

agent_point = ax.scatter(
    [], [],
    color="#F1C40F",
    edgecolors="black",
    s=95,
    zorder=5
)

ax.scatter(
    target_pos[1],
    target_pos[0],
    color="#2ECC71",
    marker="*",
    edgecolors="black",
    s=180,
    zorder=6
)

ax.legend(
    handles=[
        Patch(color="#34495E", label="Muros"),
        Patch(color="#F1C40F", label="Agente"),
        Patch(color="#E74C3C", label="Enemigos"),
        Patch(color="#3498DB", label="Ruta BFS"),
        Patch(color="#2ECC71", label="Meta")
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.02),
    ncol=3,
    fontsize=9
)

fig.tight_layout()


def update_frame(index):
    state = records[index]

    agent = state["agent"]
    enemies = state["enemies"]
    path = state["path"]

    agent_point.set_offsets(
        np.array([[agent[1], agent[0]]])
    )

    enemy_points.set_offsets(
        np.array([(col, row) for row, col in enemies])
    )

    hazards = sorted(hazard_cells(enemies))

    hazard_points.set_offsets(
        np.array(
            [(col, row) for row, col in hazards],
            dtype=float
        ).reshape(-1, 2)
    )

    if path:
        route_line.set_data(
            [col for row, col in path],
            [row for row, col in path]
        )
    else:
        route_line.set_data([], [])

    ax.set_title(
        f"Evasión dinámica con BFS — Paso {state['tick']}\n"
        f"{state['status']} | Semilla {SEED}"
    )

    return (
        route_line,
        hazard_points,
        enemy_points,
        agent_point
    )


def init_animation():
    return update_frame(0)


ani = animation.FuncAnimation(
    fig,
    update_frame,
    init_func=init_animation,
    frames=len(records),
    interval=INTERVAL_MS,
    repeat=False,
    blit=False,
    cache_frame_data=False
)

print("Simulación local calculada.")
print("Intentos de generación:", generation_attempts)
print("Corredor adicional:", corridor_added)
print("Posiciones iniciales de enemigos:", enemy_positions)
print("Pasos simulados:", records[-1]["tick"])
print("Pasos sin ruta:", blocked_ticks)
print("Cambios de ruta disponibles:", route_changes)
print("Resultado:", outcome)

# Guardar una imagen del estado final en la carpeta de Jupyter.
update_frame(len(records) - 1)
fig.savefig(
    "bfs_estado_final.png",
    dpi=200,
    bbox_inches="tight"
)

# La animación muestra estados calculados una sola vez.
html_animation = ani.to_jshtml()
plt.close(fig)
display(HTML(html_animation))

Simulación local calculada.
Intentos de generación: 1
Corredor adicional: False
Posiciones iniciales de enemigos: [(1, 29), (29, 1), (15, 14), (15, 4), (5, 5)]
Pasos simulados: 62
Pasos sin ruta: 0
Cambios de ruta disponibles: 2
Resultado: Meta alcanzada
